In [25]:
import argparse
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PERIOD = 20

In [26]:
gold_ticker = yf.Ticker("GC=F")
df = gold_ticker.history(period="max", interval='1h')

In [27]:
df = df[['Open', 'High', 'Low', 'Close']]

df.tail()

,Open,High,Low,Close
Datetime,,,,
2026-07-21 01:00:00-04:00,4054.100098,4072.800049,4051.100098,4066.000000
2026-07-21 02:00:00-04:00,4066.000000,4088.399902,4066.000000,4083.000000
2026-07-21 03:00:00-04:00,4083.300049,4084.800049,4072.100098,4075.699951
2026-07-21 04:00:00-04:00,4075.600098,4076.899902,4063.600098,4067.399902
2026-07-21 05:00:00-04:00,4067.699951,4072.800049,4065.399902,4067.500000


### Compute EMA

In [28]:
df["EMA"] = df["Low"].ewm(span=PERIOD, adjust=False).mean()

low_df = df[df['High'] < df['EMA']]

low_df["distance"] = low_df["EMA"] - low_df["Low"]

low_df.tail()

,Open,High,Low,Close,EMA,distance
Datetime,,,,,,
2026-07-16 16:00:00-04:00,3984.800049,3984.800049,3976.199951,3979.899902,4004.916427,28.716476
2026-07-16 18:00:00-04:00,3980.100098,3984.100098,3976.800049,3982.100098,4002.238677,25.438628
2026-07-16 19:00:00-04:00,3982.100098,3989.899902,3978.600098,3989.199951,3999.987383,21.387286
2026-07-16 20:00:00-04:00,3989.500000,3995.899902,3983.800049,3995.899902,3998.445732,14.645684
2026-07-16 22:00:00-04:00,3987.000000,3991.100098,3974.100098,3982.199951,3994.770321,20.670224


In [30]:
col = low_df["distance"]

# stats = {
#     "count": col.count(),
#     "mean": col.mean(),
#     "std": col.std(),
#     "min": col.min(),
#     "25%": col.quantile(0.25),
#     "median": col.median(),
#     "75%": col.quantile(0.75),
#     "max": col.max(),
# }

# print(stats)

# Or simply
print(col.describe())

count    1871.000000
mean       33.956493
std        37.694959
min         1.618960
25%        12.453122
50%        22.429425
75%        41.259686
max       365.864853
Name: distance, dtype: float64


In [31]:
import plotly.express as px

mean = col.mean()
std = col.std()

fig = px.histogram(low_df, x="distance", nbins=30)

fig.add_vline(x=mean, line_color="red", annotation_text="Mean")
fig.add_vline(x=mean - std, line_dash="dash", line_color="green")
fig.add_vline(x=mean + std, line_dash="dash", line_color="green")

fig.show()